In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import display, clear_output
import sys, pathlib

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())

def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()

print('Connected to', DB)

Connected to /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


## Database overview

In [2]:
with duckdb.connect(DB, read_only=True) as con:
    tables = [r[0] for r in con.execute("SELECT name FROM _irp_datasets ORDER BY name").fetchall()]

for t in tables:
    n = q(f'SELECT COUNT(*) FROM "{t}"').iloc[0, 0]
    print(f'  {t:<20} {n:>12,} rows')

  balance                    17,049 rows
  cashflow                   17,048 rows
  companies                   6,556 rows


  income                     17,049 rows
  industries                     74 rows
  prices                 45,386,775 rows


## Price explorer

In [3]:
all_tickers = q("SELECT DISTINCT ticker FROM prices ORDER BY ticker")["ticker"].tolist()

ticker_search = w.Combobox(
    options=all_tickers,
    value='MSFT',
    description='Ticker:',
    ensure_option=False,
    layout=w.Layout(width='200px'),
)
chart_type = w.ToggleButtons(
    options=['Candlestick', 'Close', 'OHLC'],
    value='Candlestick',
    description='Chart:',
)
date_from = w.Text(value='2020-01-01', description='From:', layout=w.Layout(width='200px'))
date_to   = w.Text(value='2025-12-31', description='To:',   layout=w.Layout(width='200px'))
btn       = w.Button(description='Load', button_style='primary')
out       = w.Output()

def load(_=None):
    with out:
        clear_output(wait=True)
        ticker = ticker_search.value.strip().upper()
        df = q("""
            SELECT date, open, high, low, close, volume
            FROM prices
            WHERE ticker = ?
              AND date >= ? AND date <= ?
            ORDER BY date
        """, [ticker, date_from.value, date_to.value])

        if df.empty:
            print(f'No data for {ticker!r} in range {date_from.value}–{date_to.value}')
            return

        fig = go.Figure()
        if chart_type.value == 'Candlestick':
            fig.add_trace(go.Candlestick(x=df['date'], open=df['open'], high=df['high'],
                                         low=df['low'], close=df['close'], name=ticker))
        elif chart_type.value == 'OHLC':
            fig.add_trace(go.Ohlc(x=df['date'], open=df['open'], high=df['high'],
                                  low=df['low'], close=df['close'], name=ticker))
        else:
            fig.add_trace(go.Scatter(x=df['date'], y=df['close'], mode='lines', name=ticker))

        fig.update_layout(
            title=f'{ticker}  ({len(df):,} days)',
            xaxis_title='Date', yaxis_title='Price',
            xaxis_rangeslider_visible=False,
            height=500,
            template='plotly_dark',
        )
        fig.show()

        # volume bar
        vfig = go.Figure(go.Bar(x=df['date'], y=df['volume'], name='Volume', marker_color='steelblue'))
        vfig.update_layout(height=200, template='plotly_dark', showlegend=False,
                           margin=dict(t=10, b=30))
        vfig.show()

btn.on_click(load)
display(w.VBox([
    w.HBox([ticker_search, date_from, date_to]),
    w.HBox([chart_type, btn]),
    out,
]))

## Fundamentals explorer

In [4]:
fund_tickers = q('SELECT DISTINCT "Ticker" FROM income WHERE "Ticker" IS NOT NULL ORDER BY "Ticker"')["Ticker"].dropna().tolist()

fund_ticker = w.Combobox(
    options=fund_tickers,
    value='MSFT' if 'MSFT' in fund_tickers else (fund_tickers[0] if fund_tickers else ''),
    description='Ticker:',
    layout=w.Layout(width='200px'),
)
fund_stmt = w.Dropdown(
    options=['income', 'balance', 'cashflow'],
    value='income',
    description='Statement:',
)
fund_btn = w.Button(description='Load', button_style='primary')
fund_out = w.Output()

def load_fund(_=None):
    with fund_out:
        clear_output(wait=True)
        ticker = fund_ticker.value.strip().upper()
        stmt   = fund_stmt.value
        df = q(f'SELECT * FROM "{stmt}" WHERE "Ticker" = ? ORDER BY "Report Date"', [ticker])
        if df.empty:
            print(f'No {stmt} data for {ticker!r}')
            return
        # show company info
        co = q('SELECT * FROM companies WHERE "Ticker" = ?', [ticker])
        if not co.empty:
            row = co.iloc[0]
            ind = q('SELECT "Industry", "Sector" FROM industries WHERE "IndustryId" = ?',
                    [int(row['IndustryId'])])
            sector = ind.iloc[0]['Sector'] if not ind.empty else 'n/a'
            industry = ind.iloc[0]['Industry'] if not ind.empty else 'n/a'
            print(f"{row['Company Name']}  |  {sector} / {industry}  |  {row['ISIN']}")
        html = df.set_index('Report Date').to_html()
        display(w.HTML(f'<div style="display:block;overflow-x:auto;overflow-y:auto;max-height:400px;width:100%">{html}</div>'))

fund_btn.on_click(load_fund)
display(w.VBox([
    w.HBox([fund_ticker, fund_stmt, fund_btn]),
    fund_out,
]))

## Companies & industries

In [5]:
sector_filter = w.Dropdown(
    options=['All'] + sorted(q('SELECT DISTINCT "Sector" FROM industries ORDER BY 1')['Sector'].tolist()),
    value='All',
    description='Sector:',
    layout=w.Layout(width='300px'),
)
co_out = w.Output()

def load_companies(change=None):
    with co_out:
        clear_output(wait=True)
        sector = sector_filter.value
        if sector == 'All':
            df = q("""
                SELECT c."Ticker", c."Company Name", i."Industry", i."Sector"
                FROM companies c
                LEFT JOIN industries i ON c."IndustryId" = i."IndustryId"
                ORDER BY c."Ticker"
            """)
        else:
            df = q("""
                SELECT c."Ticker", c."Company Name", i."Industry", i."Sector"
                FROM companies c
                LEFT JOIN industries i ON c."IndustryId" = i."IndustryId"
                WHERE i."Sector" = ?
                ORDER BY c."Ticker"
            """, [sector])
        print(f'{len(df):,} companies')
        display(df)

sector_filter.observe(load_companies, names='value')
display(w.VBox([sector_filter, co_out]))
load_companies()